In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split, StratifiedKFold, GroupShuffleSplit, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, roc_curve
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


## set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 16:02:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [ ]:
# set up config
model_train_date_str = "2024-09-01"
train_test_period_months = 12
oot_period_months = 2
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio 


pprint.pprint(config)

{'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
 'model_train_date_str': '2024-09-01',
 'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'oot_period_months': 2,
 'oot_start_date': datetime.datetime(2024, 7, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 6, 30, 0, 0),
 'train_test_period_months': 12,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)}


## get label store

In [4]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 7472
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6mob|   2023-0

In [5]:
# extract label store
labels_sdf = label_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted labels_sdf", labels_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

extracted labels_sdf 6961 2023-07-01 00:00:00 2024-08-31 00:00:00


## get features

In [6]:
# connect to feature store
folder_path = "datamart/gold/feature_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
features_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",features_store_sdf.count())

features_store_sdf.show()

row_count: 10515
+-----------+--------------------+---------------+------+--------+-------------------------+-------------+-------------+-----------------+---------------+-------------+-----------+--------------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+---------------------+-------------------+-----------------------+--------------------+---------------+-------------------------+---+-------------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------------+-------------------+--------------------+--------------------------+-----------------------+--------------------+-----------------------+-------------------+----------------------------+--------+----------------+----------------+-----------------------+
|Customer_ID|             loan_id|loan_start_date|tenure|loan_amt|Scheduled

In [7]:
# extract features from feature store
features_sdf = features_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted features_sdf", features_sdf.count(), config["train_test_start_date"], config["oot_end_date"])

extracted features_sdf 6937 2023-07-01 00:00:00 2024-08-31 00:00:00


## prepare data for modeling

In [8]:
feat = (features_store_sdf
        .withColumnRenamed("snapshot_date", "feature_snapshot_date")
        .drop("Customer_ID"))

data_pdf = labels_sdf.join(feat, on="loan_id", how="left").toPandas()

In [9]:
data_pdf

,loan_id,Customer_ID,label,label_def,snapshot_date,loan_start_date,tenure,loan_amt,Scheduled_Installment_Amt,feature_snapshot_date,...,Investment_to_Income_Ratio,Balance_to_Income_Ratio,Delayed_Payment_Rate,Credit_Utilization_Band,Credit_History_Band,Recent_Credit_Expansion_Flag,Age_Band,Occupation_Group,Clickstream_Mean,Clickstream_Vector_Norm
0,CUS_0x1037_2023_01_01,CUS_0x1037,0,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,0.0741,0.2618,3.7500,High,Long_Tenure,1,40to59,Professional,55.90,462.96
1,CUS_0x1069_2023_01_01,CUS_0x1069,0,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,0.0344,0.0906,NaN,Low,Long_Tenure,1,25to39,Professional,81.25,605.00
2,CUS_0x114a_2023_01_01,CUS_0x114a,0,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,0.0526,0.2665,1.0000,Low,Long_Tenure,1,40to59,Professional,108.35,723.95
3,CUS_0x1184_2023_01_01,CUS_0x1184,0,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,0.0168,0.2245,3.0000,Low,Long_Tenure,1,40to59,Professional,105.40,576.99
4,CUS_0x1297_2023_01_01,CUS_0x1297,1,30dpd_6mob,2023-07-01,2023-01-01,10,10000.0,0.0,2023-01-01,...,0.0110,0.0795,2.6667,Low,Established,1,40to59,Business,97.80,564.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6956,CUS_0xdf6_2023_09_01,CUS_0xdf6,0,30dpd_6mob,2024-03-01,2023-09-01,10,10000.0,0.0,2023-09-01,...,0.0886,0.1276,NaN,Medium,Long_Tenure,1,40to59,Trade,107.67,509.71
6957,CUS_0xe23_2023_09_01,CUS_0xe23,0,30dpd_6mob,2024-03-01,2023-09-01,10,10000.0,0.0,2023-09-01,...,0.0488,0.2380,14.0000,Medium,Long_Tenure,1,25to39,Creative,105.76,501.17
6958,CUS_0xe4e_2023_09_01,CUS_0xe4e,0,30dpd_6mob,2024-03-01,2023-09-01,10,10000.0,0.0,2023-09-01,...,0.0419,0.0867,NaN,Medium,Long_Tenure,1,Under25,Professional,110.10,514.69
6959,CUS_0xedd_2023_09_01,CUS_0xedd,0,30dpd_6mob,2024-03-01,2023-09-01,10,10000.0,0.0,2023-09-01,...,0.0565,0.0360,3.2500,Medium,Long_Tenure,0,40to59,Professional,111.34,519.29


In [ ]:
# split data into train - test - oot
oot_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["oot_start_date"].date()) & (data_pdf['snapshot_date'] <= config["oot_end_date"].date())]
train_test_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["train_test_start_date"].date()) & (data_pdf['snapshot_date'] <= config["train_test_end_date"].date())].reset_index(drop=True)

exclude = ['loan_id', 'snapshot_date', 'label', 'feature_snapshot_date', 'Customer_ID',
           'label_def', 'loan_start_date', 'Occupation', 'Type_of_Loan']
feature_cols = [c for c in data_pdf.columns if c not in exclude]
cat_cols = [c for c in feature_cols if data_pdf[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

X_oot = oot_pdf[feature_cols]
y_oot = oot_pdf["label"]
# stratified + grouped 80/20: first fold of a StratifiedGroupKFold (a customer is only ever on one side)
n_splits = max(2, int(round(1 / (1 - config["train_test_ratio"]))))
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=88)
train_idx, test_idx = next(
    sgkf.split(train_test_pdf, train_test_pdf["label"], groups=train_test_pdf["Customer_ID"])
)
X_train, y_train = train_test_pdf.loc[train_idx, feature_cols], train_test_pdf.loc[train_idx, "label"]
X_test,  y_test  = train_test_pdf.loc[test_idx,  feature_cols], train_test_pdf.loc[test_idx,  "label"]
groups_train = train_test_pdf.loc[train_idx, "Customer_ID"]


print('X_train', X_train.shape[0])
print('X_test', X_test.shape[0])
print('X_oot', X_oot.shape[0])
print('y_train', y_train.shape[0], round(y_train.mean(),2))
print('y_test', y_test.shape[0], round(y_test.mean(),2))
print('y_oot', y_oot.shape[0], round(y_oot.mean(),2))

X_train

In [ ]:
cat_cols

## preprocess data

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

# fit on train only; test and OOT are transformed with train statistics
transformer_stdscaler = preprocessor.fit(X_train)

X_train_processed = transformer_stdscaler.transform(X_train)
X_test_processed  = transformer_stdscaler.transform(X_test)
X_oot_processed   = transformer_stdscaler.transform(X_oot)

print('X_train_processed', X_train_processed.shape)
print('X_test_processed', X_test_processed.shape)
print('X_oot_processed', X_oot_processed.shape)

pd.DataFrame(X_train_processed, columns=transformer_stdscaler.get_feature_names_out())

## train model

In [ ]:
# Define the XGBoost classifier
xgb_clf = xgb.XGBClassifier(eval_metric='logloss', random_state=88)

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=88)

# Define the hyperparameter space to search
param_dist = {
    'n_estimators': [25, 50],
    'max_depth': [2, 3],  # lower max_depth to simplify the model
    'learning_rate': [0.01, 0.1],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# Create a scorer based on AUC score
auc_scorer = make_scorer(roc_auc_score)

# Set up the random search with cross-validation
random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    scoring="roc_auc",
    n_iter=10,  # Number of iterations for random search
    cv=cv,      # stratified, customer-grouped folds
    verbose=1,
    random_state=42,
    n_jobs=-1   # Use all available cores
)

# Perform the random search
random_search.fit(X_train_processed, y_train, groups=groups_train)

# Output the best parameters and best score
print("Best parameters found: ", random_search.best_params_)
print("Best AUC score: ", random_search.best_score_)

# Evaluate the model on the train set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_train_processed)[:, 1]
train_auc_score = roc_auc_score(y_train, y_pred_proba)
print("Train AUC score: ", train_auc_score)

# Evaluate the model on the test set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_test_processed)[:, 1]
test_auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC score: ", test_auc_score)

# Evaluate the model on the oot set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("TRAIN GINI score: ", round(2*train_auc_score-1,3))
print("Test GINI score: ", round(2*test_auc_score-1,3))
print("OOT GINI score: ", round(2*oot_auc_score-1,3))

## prepare model artefact to save

In [ ]:
model_artefact = {}

model_artefact['model'] = best_model
model_artefact['model_version'] = "credit_model_"+config["model_train_date_str"].replace('-','_')
model_artefact['preprocessing_transformers'] = {}
model_artefact['preprocessing_transformers']['stdscaler'] = transformer_stdscaler
model_artefact['data_dates'] = config
model_artefact['data_stats'] = {}
model_artefact['data_stats']['X_train'] = X_train.shape[0]
model_artefact['data_stats']['X_test'] = X_test.shape[0]
model_artefact['data_stats']['X_oot'] = X_oot.shape[0]
model_artefact['data_stats']['y_train'] = round(y_train.mean(),2)
model_artefact['data_stats']['y_test'] = round(y_test.mean(),2)
model_artefact['data_stats']['y_oot'] = round(y_oot.mean(),2)
model_artefact['results'] = {}
model_artefact['results']['auc_train'] = train_auc_score
model_artefact['results']['auc_test'] = test_auc_score
model_artefact['results']['auc_oot'] = oot_auc_score
model_artefact['results']['gini_train'] = round(2*train_auc_score-1,3)
model_artefact['results']['gini_test'] = round(2*test_auc_score-1,3)
model_artefact['results']['gini_oot'] = round(2*oot_auc_score-1,3)
model_artefact['hp_params'] = random_search.best_params_


pprint.pprint(model_artefact)

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Named DataFrame in the model's input space
feat_names = transformer_stdscaler.get_feature_names_out()
X_oot_df = pd.DataFrame(X_oot_processed, columns=feat_names, index=X_oot.index)

# 2. Exact tree SHAP (Lundberg et al., TreeSHAP) – polynomial time, no sampling
explainer = shap.TreeExplainer(best_model)          # model_output="raw" -> log-odds
sv = explainer(X_oot_df)                            # shap.Explanation: .values (n,p), .base_values, .data

# 3. Sanity check: local accuracy  f(x) = φ0 + Σ_i φ_i(x)
margin = best_model.predict(X_oot_df, output_margin=True)
assert np.allclose(sv.base_values + sv.values.sum(1), margin, atol=1e-4)

# 4. Global importance: mean |φ_i|
shap.plots.bar(sv, max_display=20)

# 5. Direction + magnitude per feature (beeswarm)
shap.plots.beeswarm(sv, max_display=20)

In [ ]:
imp = np.abs(sv.values).mean(0)                    # global: mean |φ_i|
# imp = np.abs(sv.values[i])                       # local: one prediction

order = np.argsort(imp)[::-1]
imp_sorted = imp[order]
cum = np.cumsum(imp_sorted) / imp_sorted.sum()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(imp_sorted, marker="o"); ax[0].set_title("sorted mean |SHAP|"); ax[0].set_xlabel("rank")
ax[1].plot(cum, marker="o");        ax[1].set_title("cumulative share of total |SHAP|"); ax[1].set_xlabel("rank")
plt.tight_layout(); plt.show()

In [ ]:
# Build monitoring reference data for this model version.
# Feature distributions use the training data; score distribution and ROC use OOT.
model_artefact['reference'] = {}

# 1. Numeric feature reference distributions from training data.
ref_feat = {}
for c in num_cols:
    x = pd.to_numeric(X_train[c], errors='coerce').dropna().astype(float)
    if x.empty:
        continue

    edges = np.unique(np.quantile(x, np.linspace(0, 1, 11)))
    if len(edges) < 2:
        edges = np.array([-np.inf, np.inf], dtype=float)
    else:
        edges[0] = -np.inf
        edges[-1] = np.inf

    var = x.var(ddof=1) if len(x) > 1 else np.nan
    std = x.std(ddof=1) if len(x) > 1 else np.nan
    ref_feat[c] = {
        'mean': float(x.mean()),
        'median': float(x.median()),
        'var': float(var) if pd.notna(var) else np.nan,
        'std': float(std) if pd.notna(std) else np.nan,
        'edges': edges,
        'pct': np.histogram(x, bins=edges)[0] / len(x),
        'missing_rate': float(X_train[c].isna().mean()),
    }
model_artefact['reference']['features'] = ref_feat

# 2. Reference score distribution from OOT predicted probabilities.
# These are the model's predicted probabilities, not hard class labels.
oot_scores = best_model.predict_proba(X_oot_processed)[:, 1]
score_edges = np.unique(np.quantile(oot_scores, np.linspace(0, 1, 11)))
if len(score_edges) < 2:
    score_edges = np.array([-np.inf, np.inf], dtype=float)
else:
    score_edges[0] = -np.inf
    score_edges[-1] = np.inf

model_artefact['reference']['score_edges'] = score_edges
model_artefact['reference']['score_pct'] = (
    np.histogram(oot_scores, bins=score_edges)[0] / len(oot_scores)
    if len(oot_scores) else np.array([])
)

# 3. Reference OOT ROC curve on a fixed FPR grid.
if pd.Series(y_oot).nunique() == 2 and len(oot_scores):
    fpr, tpr, _ = roc_curve(y_oot, oot_scores)
    grid = np.linspace(0, 1, 101)
    model_artefact['reference']['roc_oot'] = {
        'fpr': grid,
        'tpr': np.interp(grid, fpr, tpr),
    }
else:
    model_artefact['reference']['roc_oot'] = None

print("Reference created for:", model_artefact['model_version'])
print("Reference numeric features:", len(ref_feat))
print("Score bins:", len(model_artefact['reference']['score_pct']))

## save artefact to model bank

In [ ]:
# create model_bank dir
model_bank_directory = "model_bank/"

if not os.path.exists(model_bank_directory):
    os.makedirs(model_bank_directory)

In [ ]:
# Full path to the file
file_path = os.path.join(model_bank_directory, model_artefact['model_version'] + '.pkl')

# Write the model to a pickle file
with open(file_path, 'wb') as file:
    pickle.dump(model_artefact, file)

print(f"Model saved to {file_path}")


## test load pickle and make model inference

In [ ]:
# Load the model from the pickle file
with open(file_path, 'rb') as file:
    loaded_model_artefact = pickle.load(file)

y_pred_proba = loaded_model_artefact['model'].predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("Model loaded successfully!")

In [ ]:
import pickle

with open(file_path, 'rb') as f:
    artefact = pickle.load(f)

print(artefact['reference']['roc_oot'].keys())
# dict_keys(['fpr', 'tpr'])

print(len(artefact['reference']['roc_oot']['fpr']))
# 101

In [ ]:
import matplotlib.pyplot as plt

roc = artefact['reference']['roc_oot']

plt.plot(roc['fpr'], roc['tpr'], label='Reference OOT ROC')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

In [ ]:
# Load the saved artefact and verify inference and monitoring references.
with open(file_path, 'rb') as file:
    loaded_model_artefact = pickle.load(file)

y_pred_proba = loaded_model_artefact['model'].predict_proba(X_oot_processed)[:, 1]
loaded_oot_auc = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score:", loaded_oot_auc)
print("Model loaded successfully!")
print("Reference keys:", loaded_model_artefact['reference'].keys())
print("Reference feature count:", len(loaded_model_artefact['reference']['features']))

In [ ]:
import os
import pickle

path = "/app/model_bank/credit_model_2024_09_01.pkl"

assert "reference" in artefact, (
    "model_artefact does not contain reference. "
    "Run the reference-generation cell first."
)

with open(path, "wb") as f:
    pickle.dump(model_artefact, f)

# Verify the exact saved file
with open(path, "rb") as f:
    saved_artifact = pickle.load(f)

print("Saved path:", path)
print("Saved keys:", list(saved_artifact.keys()))
print("Reference keys:", list(saved_artifact["reference"].keys()))

In [ ]:
import pickle

with open(
    "/app/model_bank/credit_model_2024_09_01.pkl", "rb"
) as f:
    artifact = pickle.load(f)

print("Has reference:", "reference" in artifact)

In [26]:
import pandas as pd

path = (
    "datamart/gold/label_store/"
    "gold_label_store_2024_09_01.parquet"
)

labels = pd.read_parquet(path)

print("Columns:", labels.columns.tolist())
print("Rows:", len(labels))
print(labels.head(10).to_string(index=False))

print("\nSnapshot dates:")
if "snapshot_date" in labels.columns:
    print(labels["snapshot_date"].value_counts(dropna=False))

print("\nLabel definitions:")
if "label_def" in labels.columns:
    print(labels["label_def"].value_counts(dropna=False))

print("\nLoan ID suffixes:")
print(
    labels["loan_id"]
    .astype(str)
    .str.extract(r"(\d{4}_\d{2}_\d{2})$")[0]
    .value_counts(dropna=False)
)

Columns: ['loan_id', 'Customer_ID', 'label', 'label_def', 'snapshot_date']
Rows: 511
              loan_id Customer_ID  label  label_def snapshot_date
CUS_0x100b_2024_03_01  CUS_0x100b      0 30dpd_6mob    2024-09-01
CUS_0x1096_2024_03_01  CUS_0x1096      1 30dpd_6mob    2024-09-01
CUS_0x111c_2024_03_01  CUS_0x111c      1 30dpd_6mob    2024-09-01
CUS_0x112d_2024_03_01  CUS_0x112d      1 30dpd_6mob    2024-09-01
CUS_0x1204_2024_03_01  CUS_0x1204      1 30dpd_6mob    2024-09-01
CUS_0x12af_2024_03_01  CUS_0x12af      0 30dpd_6mob    2024-09-01
CUS_0x12cb_2024_03_01  CUS_0x12cb      1 30dpd_6mob    2024-09-01
CUS_0x12fc_2024_03_01  CUS_0x12fc      0 30dpd_6mob    2024-09-01
CUS_0x144c_2024_03_01  CUS_0x144c      0 30dpd_6mob    2024-09-01
CUS_0x1546_2024_03_01  CUS_0x1546      0 30dpd_6mob    2024-09-01

Snapshot dates:
snapshot_date
2024-09-01    511
Name: count, dtype: int64

Label definitions:
label_def
30dpd_6mob    511
Name: count, dtype: int64

Loan ID suffixes:
0
2024_03_01    511
N

In [27]:
import sys

print(sys.executable)
print(sys.version)

/home/airflow/.local/bin/python
3.12.9 (main, Feb  6 2025, 22:37:05) [GCC 12.2.0]


In [1]:
import glob, pickle, numpy as np, pandas as pd

art = pickle.load(open("/app/model_bank/credit_model_2024_09_01.pkl", "rb"))
ref = art["reference"]
print("edges:", np.round(ref["score_edges"], 4))
print("ref pct:", np.round(ref["score_pct"], 3))

p = pd.read_parquet("/app/datamart/gold/model_predictions/credit_model_2024_09_01/"
                    "credit_model_2024_09_01_predictions_2024_09_01.parquet")
print(p["model_predictions"].describe())
print("actual counts per ref bin:",
      np.histogram(np.clip(p["model_predictions"], ref["score_edges"][0], ref["score_edges"][-1]),
                   bins=ref["score_edges"])[0])

f = sorted(glob.glob("/app/datamart/gold/model_monitoring/credit_model_2024_09_01/covariate_2024_09_01_*.parquet"))[-1]
c = pd.read_parquet(f)
print(c.loc[c.status != "ok", ["feature", "status", "missing_rate_ref", "missing_rate_act"]])

edges: [  -inf 0.0586 0.0798 0.0996 0.1311 0.1718 0.2426 0.3533 0.517  0.7032
    inf]
ref pct: [0.101 0.1   0.1   0.1   0.1   0.101 0.1   0.1   0.1   0.101]
count    493.000000
mean       0.279543
std        0.178458
min        0.096772
25%        0.136310
50%        0.178235
75%        0.423095
max        0.725727
Name: model_predictions, dtype: float64
actual counts per ref bin: [  0   0   2 109 119  67  34  79  81   2]
                        feature              status  missing_rate_ref  \
19                         fe_1  all_values_missing               0.0   
20                         fe_2  all_values_missing               0.0   
21                         fe_3  all_values_missing               0.0   
22                         fe_4  all_values_missing               0.0   
23                         fe_5  all_values_missing               0.0   
24                         fe_6  all_values_missing               0.0   
25                         fe_7  all_values_missing           

In [2]:
   import glob, pandas as pd
   for fp in sorted(glob.glob("/app/datamart/gold/feature_store/gold_feature_store_*.parquet")):
       d = pd.read_parquet(fp)
       print(fp[-18:-8], len(d), d["fe_1"].isna().mean() if "fe_1" in d else "no fe_1")

2023_01_01 530 0.0
2023_02_01 501 0.0
2023_03_01 506 0.0
2023_04_01 510 0.0
2023_05_01 521 0.0
2023_06_01 517 0.0
2023_07_01 471 0.0
2023_08_01 481 0.0
2023_09_01 454 0.0
2023_10_01 487 0.0
2023_11_01 491 0.0
2023_12_01 489 0.0
2024_01_01 485 0.0
2024_02_01 518 0.0
2024_03_01 511 0.0
2024_04_01 513 0.0
2024_05_01 491 0.0
2024_06_01 498 0.0
2024_07_01 505 1.0
2024_08_01 543 1.0
2024_09_01 493 1.0


In [1]:
import glob, pandas as pd

# 1. where does clickstream live in each layer?
for layer in ["bronze", "silver", "gold"]:
    hits = [p for p in glob.glob(f"/app/datamart/{layer}/**/*", recursive=True)
            if "click" in p.lower() or "feature_store" in p.lower()]
    print(layer, hits[:6])

bronze ['/app/datamart/bronze/feature_clickstream', '/app/datamart/bronze/feature_clickstream/bronze_clickstream_2023_01_01.csv', '/app/datamart/bronze/feature_clickstream/bronze_clickstream_2023_02_01.csv', '/app/datamart/bronze/feature_clickstream/bronze_clickstream_2023_03_01.csv', '/app/datamart/bronze/feature_clickstream/bronze_clickstream_2023_04_01.csv', '/app/datamart/bronze/feature_clickstream/bronze_clickstream_2023_05_01.csv']
silver ['/app/datamart/silver/clickstream', '/app/datamart/silver/clickstream/silver_clickstream_2023_01_01.parquet', '/app/datamart/silver/clickstream/silver_clickstream_2023_02_01.parquet', '/app/datamart/silver/clickstream/silver_clickstream_2023_03_01.parquet', '/app/datamart/silver/clickstream/silver_clickstream_2023_04_01.parquet', '/app/datamart/silver/clickstream/silver_clickstream_2023_05_01.parquet']
gold ['/app/datamart/gold/feature_store', '/app/datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet', '/app/datamart/gold/feature_

In [2]:
import glob, pandas as pd

silver = pd.read_parquet("/app/datamart/silver/clickstream/silver_clickstream_2024_09_01.parquet")
gold = pd.read_parquet("/app/datamart/gold/feature_store/gold_feature_store_2024_09_01.parquet")

print("silver:", silver.shape, "| fe_1 null:", silver["fe_1"].isna().mean())
print("silver id sample:", silver["Customer_ID"].head(2).tolist())
print("gold:", gold.shape, "| fe_1 null:", gold["fe_1"].isna().mean())
print("gold non-fe cols:", [c for c in gold.columns if not c.startswith("fe_")])
print("gold loan_id sample:", gold["loan_id"].head(2).tolist())

# do the gold loans' customers exist in silver clickstream?
cid = (gold["Customer_ID"] if "Customer_ID" in gold
       else gold["loan_id"].str.extract(r"(CUS_0x[0-9a-f]+)")[0])
print("gold customers found in silver:", cid.isin(silver["Customer_ID"]).mean())

# is the null pattern every month, or only some?
for fp in sorted(glob.glob("/app/datamart/gold/feature_store/gold_feature_store_*.parquet")):
    d = pd.read_parquet(fp, columns=["loan_id", "fe_1"])
    print(fp[-18:-8], len(d), round(d["fe_1"].isna().mean(), 3))

silver: (8974, 22) | fe_1 null: 0.0
silver id sample: ['CUS_0x1037', 'CUS_0x1069']
gold: (493, 61) | fe_1 null: 1.0
gold non-fe cols: ['Customer_ID', 'loan_id', 'loan_start_date', 'tenure', 'loan_amt', 'Scheduled_Installment_Amt', 'snapshot_date', 'Annual_Income', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance', 'Credit_History_Age_Months', 'Age', 'Occupation', 'Clickstream_Months_Observed', 'EMI_to_Income_Ratio', 'Debt_to_Income_Ratio', 'Investment_to_Income_Ratio', 'Balance_to_Income_Ratio', 'Delayed_Payment_Rate', 'Credit_Utilization_Band', 'Credit_History_Band', 'Recent_Credit_Expansion_Flag', 'Age_Band', 'Occupation_Group', 'Clickstream_Mean', 'Clickstream_Vector_Norm']
gol

In [3]:
import glob, pandas as pd

gold = pd.read_parquet("/app/datamart/gold/feature_store/gold_feature_store_2024_09_01.parquet",
                       columns=["Customer_ID"])
loan_cust = set(gold["Customer_ID"])

pools = {}
for fp in sorted(glob.glob("/app/datamart/silver/clickstream/silver_clickstream_*.parquet")):
    pools[fp[-18:-8]] = set(pd.read_parquet(fp, columns=["Customer_ID"])["Customer_ID"])

months = sorted(pools)
print("pool size per month:", {m: len(pools[m]) for m in months[:2] + months[-2:]})
print("same pool every month:", all(pools[m] == pools[months[0]] for m in months))
print("Sept-24 loan customers found in ANY month:", len(loan_cust & set().union(*pools.values())), "of", len(loan_cust))

# coverage by month for every earlier loan cohort
for fp in sorted(glob.glob("/app/datamart/gold/feature_store/gold_feature_store_*.parquet"))[-8:]:
    ym = fp[-18:-8]
    c = set(pd.read_parquet(fp, columns=["Customer_ID"])["Customer_ID"])
    print(ym, "cohort customers in that month's clickstream:", round(len(c & pools.get(ym, set())) / len(c), 3))

pool size per month: {'2023_01_01': 8974, '2023_02_01': 8974, '2024_08_01': 8974, '2024_09_01': 8974}
same pool every month: True
Sept-24 loan customers found in ANY month: 0 of 493
2024_02_01 cohort customers in that month's clickstream: 1.0
2024_03_01 cohort customers in that month's clickstream: 1.0
2024_04_01 cohort customers in that month's clickstream: 1.0
2024_05_01 cohort customers in that month's clickstream: 1.0
2024_06_01 cohort customers in that month's clickstream: 1.0
2024_07_01 cohort customers in that month's clickstream: 0.0
2024_08_01 cohort customers in that month's clickstream: 0.0
2024_09_01 cohort customers in that month's clickstream: 0.0


In [4]:
import pandas as pd
raw = pd.read_csv("/app/data/feature_clickstream.csv", usecols=["Customer_ID"])  # adjust path
g = pd.read_parquet("/app/datamart/gold/feature_store/gold_feature_store_2024_09_01.parquet", columns=["Customer_ID"])
print(g["Customer_ID"].isin(set(raw["Customer_ID"])).mean())   # expect 0.0

0.0
